# sigmoid function — Python demo

Numerical companion to the entry [sigmoid function](https://dictionaryofml.org/terms/sigmoid.html) of the [Dictionary of Applied Machine Learning](https://dictionaryofml.org/): it recomputes what the entry states and prints one line per check.

Checks the entry's claims about the sigmoid function sigma(z) = 1 / (1 + exp(-z)): its range, limits and symmetry; the closed form of its derivative, its maximum 1/4 and its vanishing for large |z|; and the log-odds reading, where sigma of a linear function of the feature vector is exactly the posterior probability of the label when the feature vectors of the two classes are Gaussian with one shared covariance matrix. Self-contained (numpy/matplotlib only), fixed seed.

Requires NumPy and Matplotlib only, and uses fixed seeds, so the printed numbers reproduce exactly. Generated from [`pythondemos/sigmoid.py`](https://dictionaryofml.org/terms/sigmoid.py); CC BY 4.0.

In [ ]:
# Notebook shim: the script resolves output paths relative to __file__,
# which a notebook kernel does not define; everything lands in the
# working directory instead.
import os
__file__ = os.path.join(os.getcwd(), "sigmoid.py")
os.makedirs("pythondemos", exist_ok=True)

In [ ]:
"""
sigmoid.py — numerical companion to the glossary entry 'sigmoid function'.

Purpose
-------
Checks the entry's claims about the sigmoid function sigma(z) =
1 / (1 + exp(-z)): its range, limits and symmetry; the closed form of its
derivative, its maximum 1/4 and its vanishing for large |z|; and the
log-odds reading, where sigma of a linear function of the feature vector
is exactly the posterior probability of the label when the feature
vectors of the two classes are Gaussian with one shared covariance
matrix.  Self-contained (numpy/matplotlib only), fixed seed.

Blocks
------
[B-shape]   sigma maps the real line into (0, 1), tends to 0 and 1 at the
            two ends, passes through 1/2 at z = 0, is strictly increasing,
            and satisfies sigma(-z) = 1 - sigma(z).
[B-deriv]   sigma'(z) = sigma(z)(1 - sigma(z)) matches the slope of sigma
            measured between nearby points; it is at most 1/4, at z = 0,
            and at |z| = 10 it is below 1e-4: a neuron with a score of
            that size passes almost no gradient back.
[B-frost]   The translation on a real task: predict frost tomorrow at
            Krems from today's lowest temperature.  Logistic regression
            learns a score w*x + b, and the sigmoid turns it into a
            probability.  Checks that the probability falls as today's
            temperature rises, that it passes 1/2 where the score
            crosses zero, and that it beats always answering "no frost"
            in average log-loss.
[B-tmin]    Six consecutive days of the Krems record, written out for
            the entry's opening figure: the measurements a frost
            classifier reads before it produces a score.
[B-logodds] Two classes with Gaussian feature vectors sharing one
            covariance matrix: the posterior probability of the label 1
            equals sigma(a) for the log-odds a, a is a linear function
            w^T x + b of the feature vector, and log(sigma / (1 - sigma))
            recovers a.

Outputs
-------
sigmoid_curve.csv     : z, sigma(z), sigma'(z) on a grid
sigmoid_posterior.csv : feature vectors on a line through the feature
                        space, their log-odds, the posterior probability
                        of label 1 and sigma of the linear function
sigmoid_frost.csv     : today's lowest temperature in deg C and the
                        learned probability of frost tomorrow
sigmoid_tmin.csv      : day index and lowest temperature for six
                        consecutive days of the record
sigmoid.png           : matplotlib preview (checking only)
"""

import json
import urllib.request

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from pathlib import Path

OUT_DIR = Path(__file__).parent

report = []


def check(name, ok):
    report.append((name, bool(ok)))
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")


def sigma(z):
    return 1.0 / (1.0 + np.exp(-z))

**[B-shape]** sigma maps the real line into (0, 1), tends to 0 and 1 at the two ends, passes through 1/2 at z = 0, is strictly increasing, and satisfies sigma(-z) = 1 - sigma(z).

In [ ]:
z = np.linspace(-12.0, 12.0, 2401)
s = sigma(z)
check(f"[B-shape]   sigma takes values in (0, 1): on [-12, 12] between {s.min():.2e} and "
      f"{s.max():.6f}", 0.0 < s.min() and s.max() < 1.0)
check(f"[B-shape]   sigma(-40) = {sigma(-40.0):.1e} and 1 - sigma(40) = {1 - sigma(40.0):.1e}: "
      "the two limits 0 and 1", sigma(-40.0) < 1e-15 and 1 - sigma(40.0) < 1e-15)
check("[B-shape]   sigma(0) = 1/2 and sigma is strictly increasing",
      sigma(0.0) == 0.5 and bool(np.all(np.diff(s) > 0)))
check(f"[B-shape]   sigma(-z) = 1 - sigma(z), to within {np.max(np.abs(sigma(-z) - (1 - s))):.1e}",
      np.max(np.abs(sigma(-z) - (1 - s))) < 1e-15)

**[B-deriv]** sigma'(z) = sigma(z)(1 - sigma(z)) matches the slope of sigma measured between nearby points; it is at most 1/4, at z = 0, and at |z| = 10 it is below 1e-4: a neuron with a score of that size passes almost no gradient back.

In [ ]:
ds = s * (1.0 - s)
slope = (sigma(z + 1e-5) - sigma(z - 1e-5)) / 2e-5
check(f"[B-deriv]   sigma'(z) = sigma(z)(1 - sigma(z)) matches the measured slope to within "
      f"{np.max(np.abs(ds - slope)):.1e}", np.max(np.abs(ds - slope)) < 1e-8)
check(f"[B-deriv]   the derivative is at most 1/4, reached at z = {z[np.argmax(ds)]:.2f}",
      abs(ds.max() - 0.25) < 1e-12 and z[np.argmax(ds)] == 0.0)
check(f"[B-deriv]   at |z| = 10 the derivative is {sigma(10.0) * (1 - sigma(10.0)):.1e}: almost "
      "no gradient passes back", sigma(10.0) * (1 - sigma(10.0)) < 1e-4)
with open(OUT_DIR / "sigmoid_curve.csv", "w") as fh:
    fh.write("z,sigma,derivative\n")
    for zi, si, di in zip(z[::20], s[::20], ds[::20]):
        fh.write(f"{zi:.2f},{si:.6f},{di:.6f}\n")

**[B-logodds]** Two classes with Gaussian feature vectors sharing one covariance matrix: the posterior probability of the label 1 equals sigma(a) for the log-odds a, a is a linear function w^T x + b of the feature vector, and log(sigma / (1 - sigma)) recovers a.

In [ ]:
# two classes in the plane, Gaussian feature vectors with one shared
# covariance matrix; the posterior probability of label 1 follows from the
# two densities and the class probabilities
rng = np.random.default_rng(0)
mu1, mu0 = np.array([1.5, 0.5]), np.array([-1.0, -0.5])
C = np.array([[1.0, 0.4], [0.4, 0.8]]); Cinv = np.linalg.inv(C)
p1 = 0.4                                            # probability of the label 1


def density(x, mu):
    d = x - mu
    return np.exp(-0.5 * np.einsum("ij,jk,ik->i", d, Cinv, d)) / (2 * np.pi * np.sqrt(np.linalg.det(C)))


w = Cinv @ (mu1 - mu0)
b = -0.5 * mu1 @ Cinv @ mu1 + 0.5 * mu0 @ Cinv @ mu0 + np.log(p1 / (1 - p1))
xs = rng.standard_normal((2000, 2)) * 1.5
logodds = np.log(density(xs, mu1) * p1) - np.log(density(xs, mu0) * (1 - p1))
posterior = density(xs, mu1) * p1 / (density(xs, mu1) * p1 + density(xs, mu0) * (1 - p1))
check(f"[B-logodds] the posterior probability of the label 1 equals sigma(a) for the log-odds a, "
      f"to within {np.max(np.abs(posterior - sigma(logodds))):.1e}",
      np.max(np.abs(posterior - sigma(logodds))) < 1e-12)
check(f"[B-logodds] with one shared covariance matrix the log-odds is linear, a = w^T x + b with "
      f"w = ({w[0]:.2f}, {w[1]:.2f}), b = {b:.2f}, to within "
      f"{np.max(np.abs(logodds - (xs @ w + b))):.1e}", np.max(np.abs(logodds - (xs @ w + b))) < 1e-10)
check("[B-logodds] log(sigma / (1 - sigma)) recovers the log-odds",
      np.max(np.abs(np.log(posterior / (1 - posterior)) - logodds)) < 1e-8)
t = np.linspace(-3.0, 3.0, 121)
line = np.outer(t, (mu1 - mu0) / np.linalg.norm(mu1 - mu0)) + (mu1 + mu0) / 2
post_line = density(line, mu1) * p1 / (density(line, mu1) * p1 + density(line, mu0) * (1 - p1))
with open(OUT_DIR / "sigmoid_posterior.csv", "w") as fh:
    fh.write("t,x1,x2,logodds,posterior,sigma_linear\n")
    for ti, xi, lo, po in zip(t, line, line @ w + b, post_line):
        fh.write(f"{ti:.2f},{xi[0]:.4f},{xi[1]:.4f},{lo:.4f},{po:.6f},{sigma(lo):.6f}\n")

**[B-frost]** The translation on a real task: predict frost tomorrow at Krems from today's lowest temperature. Logistic regression learns a score w*x + b, and the sigmoid turns it into a probability. Checks that the probability falls as today's temperature rises, that it passes 1/2 where the score crosses zero, and that it beats always answering "no frost" in average log-loss.

In [ ]:
# What the translation is for: a score is unbounded, a forecast must be a
# probability.  Krems 2024, feature = today's lowest temperature, label = frost
# tomorrow (next day's lowest below zero).
URL = ("https://dataset.api.hub.geosphere.at/v1/station/historical/"
       "klima-v2-1d?parameters=tlmin&station_ids=3805"
       "&start=2024-01-01&end=2024-12-31")
with urllib.request.urlopen(URL, timeout=120) as resp:
    payload = json.load(resp)
tlmin = np.array(
    payload["features"][0]["properties"]["parameters"]["tlmin"]["data"],
    dtype=float)
x_frost = tlmin[:-1]                       # today's lowest temperature
y_frost = (tlmin[1:] < 0.0).astype(float)  # 1 if frost tomorrow, else 0
check("[B-frost] 365 days, and both labels occur",
      len(x_frost) == 365 and 0 < y_frost.sum() < 365)

# logistic regression on the single feature, by gradient descent
w_frost, b_frost = 0.0, 0.0
for _ in range(20000):
    p = sigma(w_frost * x_frost + b_frost)
    w_frost -= 0.05 * float(((p - y_frost) * x_frost).mean())
    b_frost -= 0.05 * float((p - y_frost).mean())
prob = lambda degrees: float(sigma(w_frost * degrees + b_frost))
check("[B-frost] the probability of frost falls as today gets warmer", w_frost < 0)
cross = -b_frost / w_frost                             # the score crosses zero here
check("[B-frost] the probability passes 1/2 where the score crosses zero",
      abs(prob(cross) - 0.5) < 1e-6)
eps = 1e-12
fitted = float(-(y_frost * np.log(sigma(w_frost * x_frost + b_frost) + eps)
                 + (1 - y_frost) * np.log(1 - sigma(w_frost * x_frost + b_frost) + eps)).mean())
base_p = float(y_frost.mean())
constant = float(-(y_frost * np.log(base_p + eps)
                   + (1 - y_frost) * np.log(1 - base_p + eps)).mean())
check("[B-frost] it beats always answering with the base rate", fitted < constant)
print(f"  [B-frost] frost follows {int(y_frost.sum())} of 365 days; the "
      f"learned score crosses zero at {cross:.1f} deg C, where the sigmoid "
      f"gives 1/2")
print(f"  [B-frost] probability of frost tomorrow: {prob(-5.0):.2f} after a "
      f"night at -5 deg C, {prob(0.0):.2f} at 0, {prob(5.0):.2f} at +5; "
      f"average logloss {fitted:.3f} against {constant:.3f} for the base rate")
grid = np.linspace(x_frost.min(), x_frost.max(), 160)
with open(OUT_DIR / "sigmoid_frost.csv", "w") as fh:
    fh.write("tmin,prob\n")
    for temp in grid:
        fh.write(f"{temp:.3f},{prob(float(temp)):.6f}\n")

**[B-tmin]** Six consecutive days of the Krems record, written out for the entry's opening figure: the measurements a frost classifier reads before it produces a score.

In [ ]:
# The six days the entry opens on: what the classifier reads, before any
# score or probability enters.
FIRST = 10                                  # 11-16 January 2024
six = tlmin[FIRST:FIRST + 6]
six_dates = [d[8:10] + "/" + d[5:7] for d in
             payload["timestamps"][FIRST:FIRST + 6]]
check("[B-tmin] six consecutive days, all below the January maximum",
      len(six) == 6 and six.max() < 15.0)
# the last of the six is the day the entry's figure scores
last_score = float(w_frost * six[-1] + b_frost)
last_prob = prob(six[-1])
check("[B-tmin] the last day scores above zero, so a warning is issued",
      last_score > 0.0 and last_prob > 0.5)
with open(OUT_DIR / "sigmoid_tmin.csv", "w") as fh:
    fh.write("day,tmin,date\n")
    for i, (value, when) in enumerate(zip(six, six_dates), start=1):
        fh.write(f"{i},{value:.1f},{when}\n")
print("  [B-tmin] six days " + six_dates[0] + " to " + six_dates[-1] + ": "
      + ", ".join(f"{v:.1f}" for v in six) + " deg C")
print(f"  [B-tmin] the last day scores h(x) = {last_score:.2f}, which the "
      f"sigmoid turns into {last_prob:.0%} chance of frost the next day")

# -------------------------------------------------------------- preview
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13.5, 3.6))
ax1.plot(z, s, "k-", label=r"$\sigma(z)$")
ax1.plot(z, ds, "k--", label=r"$\sigma'(z) = \sigma(z)(1-\sigma(z))$")
ax1.axhline(0.25, color="0.6", lw=0.8, ls=":", label="1/4")
ax1.set_xlabel("score $z$"); ax1.set_ylabel("value"); ax1.set_title("the sigmoid function and its derivative")
ax1.legend(frameon=False, fontsize=8)
ax2.plot(t, post_line, "k-", lw=2.5, alpha=0.4, label="posterior probability of label 1")
ax2.plot(t, sigma(line @ w + b), "k--", label=r"$\sigma(\mathbf{w}^\top\mathbf{x}+b)$")
ax2.set_xlabel("position along the line between the class means"); ax2.set_ylabel("probability")
ax2.set_title("two Gaussian classes, one covariance matrix"); ax2.legend(frameon=False, fontsize=8)
ax3.plot(x_frost, y_frost, "o", ms=3, mfc="none", color="0.5",
         label="day (1 = frost tomorrow)")
ax3.plot(grid, [prob(float(g)) for g in grid], "k-",
         label=r"$\sigma(wx+b)$, learned")
ax3.axhline(0.5, color="0.6", lw=0.8, ls=":")
ax3.set_xlabel("lowest temperature today (deg C)")
ax3.set_ylabel("probability of frost tomorrow")
ax3.set_title("a score turned into a probability")
ax3.legend(frameon=False, fontsize=8, loc="center left")
fig.tight_layout()
fig.savefig(OUT_DIR / "sigmoid.png", dpi=110)

n_ok = sum(ok for _, ok in report)
print(f"\n{n_ok}/{len(report)} checks pass")
print(f"wrote {OUT_DIR / 'sigmoid_curve.csv'}, {OUT_DIR / 'sigmoid_posterior.csv'}, {OUT_DIR / 'sigmoid.png'}")
if n_ok != len(report):
    raise SystemExit(1)